# Deteccao de Malware com GANs — Aula 8: Consolidacao multi-dataset em 3 datasets REAIS

**Objetivo:** unir os resultados das aulas 6 (UNSW-NB15), 7 (CICIDS2017) e `aula_iot23_real`
(IoT-23 real, capturas Stratosphere) — os 3 datasets sao **reais** — e gerar a tabela e as figuras do artigo.

O artigo **"Reducing False Negatives in IoT Malware Detection"** agora usa **somente dados reais**:
nao ha mais amostra sintetica nem proxy.

**Protocolo identico nos 3 datasets:** 40.000 fluxos (82% benigno / 18% ataque), split 70/30 estratificado
(semente 42), `StandardScaler` + `SelectPercentile(60%)`, 6 cenarios de balanceamento
(Original, SMOTETomek, GAN+MLP, WGAN-GP, cWGAN-GP, CTGAN) x 4 classificadores (MLP, XGBoost, RandomForest, LSTM).

**Criterio da tabela do artigo (F1-guard):** para cada modelo/dataset escolhe-se o cenario balanceado de
**menor FN** (ataque) / **menor FP** (benigno) cujo F1 fique dentro de **0.05** do melhor F1 balanceado
do modelo — evita escolher um cenario que reduza FN colapsando a precisao (ex.: GAN+MLP no LSTM).


In [ ]:
# ========================================
# CELULA 1: Instalacoes + seeds
# ========================================
!pip install -q matplotlib seaborn pandas numpy

import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
print("OK - ambiente pronto.")


In [ ]:
# ========================================
# CELULA 2: Upload dos CSVs (Colab) ou diretorio local
# ========================================
REQUERIDOS = ["resultados_unsw_nb15.csv", "resultados_cicids2017.csv", "resultados_iot23_real.csv"]

def locais():
    return sorted(glob.glob("resultados_*.csv") + glob.glob("C:/Users/Carlos/Desktop/Novo_Artigo/resultados_*.csv"))

def upload():
    try:
        from google.colab import files
        up = files.upload()
        print("Upload concluido:", [f for f in up.keys()])
    except Exception:
        print("Nao e Colab. Coloque os CSVs na pasta do notebook e execute Novamente.")

caminhos = locais()
print("CSVs encontrados na pasta atual:")
for p in caminhos:
    print(" ", os.path.basename(p), f"({os.path.getsize(p)/1024:.0f} KB)")

faltantes = [r for r in REQUERIDOS if r not in [os.path.basename(p) for p in caminhos]]
while faltantes:
    print(f"\nFALTANDO: {faltantes}. Faca upload dos arquivos obrigatorios.")
    upload()
    caminhos = locais()
    faltantes = [r for r in REQUERIDOS if r not in [os.path.basename(p) for p in caminhos]]

print("Todos os CSVs obrigatorios presentes (3 datasets reais).")


In [ ]:
# ========================================
# CELULA 3: Concatenar os CSVs reais + nomes limpos
# ========================================
csvs = sorted(set(glob.glob("resultados_*.csv") + glob.glob("C:/Users/Carlos/Desktop/Novo_Artigo/resultados_*.csv")))
print("CSVs lidos:", [os.path.basename(c) for c in csvs])
df = pd.concat([pd.read_csv(c) for c in csvs], ignore_index=True)
print(f"Total de linhas: {len(df)}")

df['Dataset'] = df['Dataset'].astype(str).str.strip()
df['Cenario'] = df['Cenario'].astype(str).str.strip()
df['Modelo']  = df['Modelo'].astype(str).str.strip()

# exclui qualquer resto de proxy/sintetico (defensivo - os notebooks reais geram "IoT-23 REAL")
antes = len(df)
df = df[~df['Dataset'].str.contains("proxy|sintetico|sintético", case=False, na=False)]
removidas = antes - len(df)
if removidas:
    print(f"(Linhas de proxy/sintetico removidas: {removidas})")

# normaliza o nome do dataset real para o artigo
df['Dataset'] = df['Dataset'].replace({"IoT-23 REAL": "IoT-23", "IoT-23 Sintetico": "IoT-23"})

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
print("\nTODAS AS LINHAS:")
print(df.to_string(index=False))

print("\nF1 medio por dataset e modelo:")
resumo = df.groupby(["Dataset", "Modelo"])["F1-Score"].mean().unstack().round(4)
print(resumo.to_string())


In [ ]:
# ========================================
# CELULA 4: Tabela do artigo — F1-guard (menor FN/FP sem colapsar precisao)
# ========================================
ORIG = "Original (sem balancear)"
TOL_F1 = 0.05

def seleciona(sub, alvo, asc=1):
    """Cenario balanceado com menor {alvo} entre os com F1 >= melhor_f1 - TOL_F1."""
    if len(sub) == 0:
        return None
    best_f1 = sub["F1-Score"].max()
    cand = sub[sub["F1-Score"] >= best_f1 - TOL_F1]
    if len(cand) == 0:
        cand = sub
    return cand.sort_values([alvo, "F1-Score"], ascending=[asc, False]).iloc[0]

tabela_rows = []
for dataset in df["Dataset"].unique():
    for modelo in ["MLP", "XGBoost", "RandomForest", "LSTM"]:
        sub = df[(df["Dataset"] == dataset) & (df["Modelo"] == modelo)]
        if sub.empty:
            continue
        base = sub[sub["Cenario"] == ORIG].iloc[0]["FN"] if (sub["Cenario"] == ORIG).any() else np.nan
        bal = sub[sub["Cenario"] != ORIG]
        att = seleciona(bal, "FN")
        ben = seleciona(bal, "FP")
        if att is None or ben is None:
            continue
        red = (100 * (1 - att["FN"] / base)) if base and base > 0 else 0.0
        tabela_rows.append({
            "Dataset": dataset, "Modelo": modelo,
            "TP_att": int(att["TP"]), "FN_att": int(att["FN"]),
            "FP_att": int(att["FP"]), "TN_att": int(att["TN"]),
            "Cenario_att": att["Cenario"], "F1_att": round(att["F1-Score"], 4),
            "FN_orig_att": int(base) if not np.isnan(base) else "",
            "Red_FN_pct_att": round(red, 1),
            "TP_ben": int(ben["TP"]), "FN_ben": int(ben["FN"]),
            "FP_ben": int(ben["FP"]), "TN_ben": int(ben["TN"]),
            "Cenario_ben": ben["Cenario"], "F1_ben": round(ben["F1-Score"], 4),
        })

df_tab = pd.DataFrame(tabela_rows)
pd.set_option("display.width", 240)
print("=" * 150)
print("  TABELA DO ARTIGO (F1-guard: menor FN/FP com F1 >= melhor - 0.05)")
print("=" * 150)
print(df_tab.to_string(index=False))
df_tab.to_csv("tabela1_artigo.csv", index=False)
print("\nSalvo: tabela1_artigo.csv")

print("\nREDUCAO DE FN vs ORIGINAL (cenario escolhido por F1-guard):")
for modelo in ["MLP", "XGBoost", "RandomForest", "LSTM"]:
    t = df_tab[df_tab["Modelo"] == modelo]
    for _, r in t.iterrows():
        print(f"  {r['Dataset']:12s} {modelo:12s} | FN {int(r['FN_orig_att']):5d} -> {r['FN_att']:5d} "
              f"({r['Red_FN_pct_att']:+7.1f}%) | FP {r['FP_att']:4d} | F1 {r['F1_att']:.4f} | {r['Cenario_att']}")


In [ ]:
# ========================================
# CELULA 5: Figuras do artigo (dados reais)
# ========================================
cenarios = ["Original (sem balancear)", "SMOTETomek", "GAN+MLP", "WGAN-GP", "cWGAN-GP", "CTGAN"]
cen_simples = ["Original", "SMOTETomek", "GAN+MLP", "WGAN-GP", "cWGAN-GP", "CTGAN"]
cores = dict(zip(cenarios, ["#bdbdbd", "#7b1fa2", "#8e24aa", "#6a1b9a", "#4a148c", "#2e7d32"]))
cores["Original (sem balancear)"] = "#bdbdbd"
modelos = ["MLP", "XGBoost", "RandomForest", "LSTM"]
cen_ord = [c for c in cenarios if c in set(df["Cenario"])]

datasets = df["Dataset"].unique()
fig, axes = plt.subplots(1, len(datasets), figsize=(7 * len(datasets), 6), sharey=False)
if len(datasets) == 1: axes = [axes]
for ax, ds in zip(axes, datasets):
    sub = df[df["Dataset"] == ds]
    x = np.arange(len(modelos)); w = 0.13
    for i, cenario in enumerate(cen_ord):
        vals = []
        for mdl in modelos:
            v = sub[(sub["Cenario"] == cenario) & (sub["Modelo"] == mdl)]["FN"].values
            vals.append(int(v[0]) if len(v) > 0 else 0)
        ax.bar(x + i * w, vals, w, label=cenario, color=cores[cenario], edgecolor="black", lw=0.3)
    ax.set_xticks(x + w * 2.5)
    ax.set_xticklabels(modelos)
    ax.set_title(f"{ds}", fontweight="bold")
    ax.set_ylabel("FN (ataques escaparam)")
    ax.legend(fontsize=7)
plt.suptitle("Figura 3 - Reducao de FN por modelo (dados reais)", fontweight="bold", y=1.0)
plt.tight_layout()
plt.show()

pivot = df.pivot_table(values="F1-Score", index=["Dataset", "Modelo"], columns="Cenario", aggfunc="mean")
pivot = pivot.reindex(columns=cen_ord)
plt.figure(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", linewidths=0.5,
            cbar_kws={"label": "F1-Score"})
plt.title("Figura 2 - F1-Score por dataset, modelo e cenario (dados reais)", fontweight="bold")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nMELHORES POR DATASET/MODELO (F1-guard):")
print(df_tab.sort_values(["Dataset", "Modelo"])[["Dataset", "Modelo", "FN_orig_att", "FN_att", "Cenario_att"]].to_string(index=False))


## 9. Discussao consolidada (dados 100% reais)

**Leitura do artigo:**

- **FN (foco principal):** o FN e o erro mais caro — malware que escapa da deteccao. A tabela mostra a
  reducao de FN por modelo/dataset quando se aplica a melhor estrategia balanceada (com F1-guard).

- **Achado transversal mais importante:** o **LSTM** e o classificador mais degradado pelo desbalanceamento
  (F1 de 0.33 no CICIDS2017 e 0.71 no UNSW-NB15 em baseline) e o que mais se beneficia do balanceamento em
  **todos os 3 datasets reais**: FN reduzido em ~83% (CICIDS2017), ~81% (UNSW-NB15) e ~41% (IoT-23).

- **Arvores (XGBoost/RandomForest) sao quase imunes:** ja dominam o baseline desbalanceado (por ex. XGBoost
  perde apenas 4 ataques no CICIDS2017 sem balancear), e o oversampling agrega pouco ou nada.

- **MLP e inconsistente:** no CICIDS2017 o baseline ja e excelente (37 FN) e o balanceamento so adiciona
  ruido; no UNSW-NB15 ha pequeno ganho; no IoT-23 real e neutro. Isso e reportado honestamente no artigo.

- **Nenhuma familia de gerador domina:** as vezes o SMOTETomek (classico) vence, as vezes WGAN-GP/cWGAN-GP
  (generativo). CTGAN continua sendo opcao solida (especialmente CICIDS/LSTM). O artigo nao afirma um vencedor
  absoluto — afirma que o balanceamento ajuda principalmente o modelo sequencial profundo.

**Limitacoes declaradas no artigo:** os 3 datasets sao publicos e reais (IoT-23 e um subconjunto das capturas
Stratosphere via HuggingFace; features do Zeek conn.log no IoT-23 vs. features de fluxo no UNSW/CICIDS).


## 10. Conclusao e proximos passos

- **Todos os 3 datasets sao reais**: `resultados_unsw_nb15.csv`, `resultados_cicids2017.csv`,
  `resultados_iot23_real.csv` + `tabela1_artigo.csv`.

- O artigo foi reescrito sem nenhuma mencao a amostra sintetica/proxy: os numeros de abstract, resultados e
  conclusao agora correspondem aos CSVs desta pasta.

- Para submeter: gere o PDF no Overleaf com os arquivos de `artigo_iot23` (main.tex + figuras +
  references.bib). A tabela `tabela1_artigo.csv` pode ser incorporada como Tabela II.
